# 1- Marge

Rappelons que la marge d’un point donné est définie comme suit :
$$\gamma(x, y, \theta, \theta_0) = \frac{y(\theta \cdot x + \theta_0)}{\|\theta\|}$$

Le problème : Un séparateur peut avoir une marge immense pour $99\%$ des points, mais passer à $0{,}0001$ millimètre d'un point critique (ou même mal le classer). Évaluer un séparateur sur un seul point ne donne aucune garantie sur sa qualité globale.

Par conséquent, nous souhaitons trouver une fonction de score $S$ pour un séparateur $(\theta, \theta_0)$, de telle manière que la maximisation de *S* permette d’obtenir un meilleur séparateur.

### 1. Le critere de la somme des marge (Margin Inovera)
Marge Inovera (dérivé de « Margin over all » (la marge sur l'ensemble des données)) suggère que, puisque de grandes marge bénéficiaires sont souhaitables, nous devrions maximiser la somme de toutes ces marge bénéficiaires. Elle définit donc cela comme suit :
$$S_{\text{sum}}(\theta, \theta_0) = \sum_{i} \gamma(x^{(i)}, y^{(i)}, \theta, \theta_0)$$

### 2. Le critère de la Marge Minimale (Minnie Malle)

Pour éviter que des points très éloignés ne masquent de mauvais classements, on définit le score du séparateur par sa **marge minimale** (le pire cas) :

$$S_{\text{min}}(\theta, \theta_0) = \min_{i} \gamma(x^{(i)}, y^{(i)}, \theta, \theta_0)$$

* **Objectif :** Maximiser $S_{\text{min}}$ revient à éloigner la frontière le plus possible des points les plus proches des deux classes.
* **Résultat :** C'est le principe fondateur des **SVM (Support Vector Machines)** et des classifieurs à marge maximale.

### 3. Le critère de la Marge Maximale (Maxim Argent)

Maxim Argent propose de définir le score par la **marge maximale** :

$$S_{\text{max}}(\theta, \theta_0) = \max_{i} \gamma(x^{(i)}, y^{(i)}, \theta, \theta_0)$$

* **Problème :** C'est un anti-exemple. Ce score ignore totalement les erreurs commises sur l'ensemble du jeu de données tant qu'au moins un point est très éloigné du bon côté.

Examinons les données suivantes, ainsi que les deux séparateurs possibles (le rouge et le bleu).

```python

data = np.array([[1, 2, 1, 2, 10, 10.3, 10.5, 10.7],
                 [1, 1, 2, 2,  2,  2,  2, 2]])
labels = np.array([[-1, -1, 1, 1, 1, 1, 1, 1]])
blue_th = np.array([[0, 1]]).T
blue_th0 = -1.5
red_th = np.array([[1, 0]]).T
red_th0 = -2.5


```

In [1]:
import numpy as np

In [12]:
data = np.array([[1, 2, 1, 2, 10, 10.3, 10.5, 10.7], [1, 1, 2, 2,  2,  2,  2, 2]])

labels = np.array([[-1, -1, 1, 1, 1, 1, 1, 1]])

blue_th = np.array([[0, 1]]).T
blue_th0 = -1.5

red_th = np.array([[1, 0]]).T
red_th0 = -2.5


In [16]:
data.shape

(2, 8)

In [17]:
red_th.shape

(2, 1)

Quelles sont les valeurs de chaque score $(S_{\text{sum}}, S_{\text{min}}, S_{{\text{max}}})$

In [18]:
# Ssum
s_sum_blue = np.sum((labels * ((blue_th.T@data) + blue_th0)) / np.linalg.norm(blue_th))
s_sum_red  = np.sum((labels * ((red_th.T@data) + red_th0)) / np.linalg.norm(red_th))

# Smin
s_min_blue = np.min((labels * ((blue_th.T@data) + blue_th0)) / np.linalg.norm(blue_th))
s_min_red  = np.min((labels * ((red_th.T@data) + red_th0)) / np.linalg.norm(red_th))

# Smax
s_max_blue = np.max((labels * ((blue_th.T@data) + blue_th0)) / np.linalg.norm(blue_th))
s_max_red  = np.max((labels * ((red_th.T@data) + red_th0)) / np.linalg.norm(red_th))

print(f"Ssum blue = {s_sum_blue}")
print(f"Smin blue = {s_min_blue}")
print(f"Smax blue = {s_max_blue}")

print(f"Ssum red = {s_sum_red}")
print(f"Smin red = {s_min_red}")
print(f"Smax blue = {s_max_red}")

print(f"Blue: [{s_sum_blue}, {s_min_blue}, {s_max_blue}]")
print(f"Red: [{s_sum_red}, {s_min_red}, {s_max_red}]")


Ssum blue = 4.0
Smin blue = 0.5
Smax blue = 0.5
Ssum red = 31.5
Smin red = -1.5
Smax blue = 8.2
Blue: [4.0, 0.5, 0.5]
Red: [31.5, -1.5, 8.2]


![1A](./assets/q1A.png)

```python
[31.5, -1.5, 8.2]
```

![1BQ](./assets/q1B.png)

```python
[4.0, 0.5, 0.5]
```

![1CQ](./assets/q1C.png)

red

![1DQ](./assets/q1D.png)

blue

![1EQ](./assets/q1E.png)

red

![1FQ](./assets/q1F.png)

$S_{\text{min}}$

# 2. Perte (Loss)

Sur la base de ce qui a été dit précédemment, nous avons décidé d’essayer de trouver un séparateur linéaire $(\theta, \theta_0)$ qui maximise la marge minimale (c’est-à-dire la distance entre le séparateur et les points qui lui sont les plus proches). Nous définissons la marge d’un ensemble de données $(X, Y)$ par rapport à un séparateur comme étant :

$$\gamma(X, Y, \theta, \theta_0) = \min_{i=1, \dots, n} \frac{y^{(i)}(\theta^T x^{(i)} + \theta_0)}{\|\theta\|}$$

ou sous forme condensée :

$$\gamma(X, Y, \theta, \theta_0) = \min_{i=1, \dots, n} \gamma(x^{(i)}, y^{(i)}, \theta, \theta_0)$$

une façon de résoudre ce problème consiste à définir une valeur $\gamma_{\text{ref}}$ pour la marge du jeu de données, puis à chercher un séparateur linéaire qui maximise la valeur $\gamma_{\text{ref}}$

![2A](./assets/2AR.png)

![2B](./assets/2B.png)

![2C](./assets/2C.png)

Nous voulons maintenant améliorer cette marge de garantie, qui est extrêmement faible, dans l’algorithme de Perceptron. Comme nous l’avons vu lors du cours, une manière efficace de concevoir des algorithmes d’apprentissage consiste à les présenter comme des problèmes d’optimisation, puis à utiliser des stratégies d’optimisation de nature générale pour les résoudre.

Une forme typique du problème d’optimisation consiste à minimiser un objectif dont la forme est la suivante :

$$J(\theta, \theta_0) = \frac{1}{n} \sum_{i=1}^{n} L(x^{(i)}, y^{(i)}, \theta, \theta_0) + \lambda R(\theta, \theta_0)$$

![note](./assets/note.png)

![2D](./assets/2D.png)

![2E](./assets/2E.png)

# 3. Simply inseparable 

Nous préférerions une fonction de perte qui aide à orienter le processus d’optimisation vers une solution adéquate, surtout dans le cas où les données sont linéairement séparables. De plus, dans les ensembles de données réels, il est relativement rare que les données soient linéairement séparables. Par conséquent, notre algorithme doit être capable de gérer ce cas également, tout en cherchant à trouver un séparateur linéaire optimal, même s’il n’est pas parfait. Au lieu d’utiliser la fonction de perte $(0, \infty)$ , nous devrions concevoir une fonction de perte qui nous permette de relâcher la contrainte selon laquelle tous les points doivent respecter un certain écart minimal par rapport au séparateur linéaire, tout en continuant à favoriser des écarts importants entre les points.

![loss](./assets/image.png)

![3A](./assets/3A.png)

```python
data = np.array([[1.1, 1, 4],[3.1, 1, 2]])
labels = np.array([[1, -1, -1]])
th = np.array([[1, 1]]).T
th0 = -4
```

![3B](./assets/3B.png)

data = np.array([[1.1, 1, 4],[3.1, 1, 2]])
labels = np.array([[1, -1, -1]])
th = np.array([[1, 1]]).T
th0 = -4

In [27]:
y_ref = np.sqrt(2) / 2

In [32]:
margin = (labels * ((th.T@data) + th0)) / np.linalg.norm(th)
lh = np.where(y_ref > margin, 1 - (margin/y_ref), 0)

In [33]:
lh

array([[0.8, 0. , 3. ]])

In [34]:
import sympy as sp

In [35]:
exact = np.vectorize(sp.nsimplify)(lh)

print(exact)

[[4/5 0 3]]


![3B1](./assets/3B1.png)

---